Q1

In [2]:
class Node:
    def __init__(self, name, value=None, children=None):
        self.name = name
        self.value = value
        self.children = children or []

def minimax(node, depth, is_max, visited):
    visited.append(node.name)
    if depth == 0 or not node.children:
        return node.value, visited

    if is_max:
        best_val = float('-inf')
        for child in node.children:
            val, _ = minimax(child, depth - 1, False, visited)
            best_val = max(best_val, val)
        node.value = best_val
        return best_val, visited
    else:
        best_val = float('inf')
        for child in node.children:
            val, _ = minimax(child, depth - 1, True, visited)
            best_val = min(best_val, val)
        node.value = best_val
        return best_val, visited

leaf1 = Node("D", 3)
leaf2 = Node("E", 5)
leaf3 = Node("F", 2)
leaf4 = Node("G", 9)
node_b = Node("B", children=[leaf1, leaf2])
node_c = Node("C", children=[leaf3, leaf4])
root = Node("A", children=[node_b, node_c])

val, order = minimax(root, 3, True, [])
print(f"Minimax Value: {val}")
print(f"Visit Order: {order}")

val_limited, _ = minimax(root, 1, True, [])
print(f"Depth-Limited (Depth 1) Value: {val_limited}")

Minimax Value: 3
Visit Order: ['A', 'B', 'D', 'E', 'C', 'F', 'G']
Depth-Limited (Depth 1) Value: 3


Q2

In [3]:
def alpha_beta(node, depth, alpha, beta, is_max, pruned):
    if depth == 0 or not node.children:
        return node.value

    if is_max:
        best_val = float('-inf')
        for child in node.children:
            val = alpha_beta(child, depth - 1, alpha, beta, False, pruned)
            best_val = max(best_val, val)
            alpha = max(alpha, best_val)
            if beta <= alpha:
                pruned.append(child.name)
                break
        node.value = best_val
        return best_val
    else:
        best_val = float('inf')
        for child in node.children:
            val = alpha_beta(child, depth - 1, alpha, beta, True, pruned)
            best_val = min(best_val, val)
            beta = min(beta, best_val)
            if beta <= alpha:
                pruned.append(child.name)
                break
        node.value = best_val
        return best_val

pruned_nodes = []
final_val = alpha_beta(root, 3, float('-inf'), float('inf'), True, pruned_nodes)
print(f"Minimax Value with Pruning: {final_val}")
print(f"Pruned Nodes: {pruned_nodes}")

Minimax Value with Pruning: 3
Pruned Nodes: ['F']


Q3

In [4]:
leaf5 = Node("H", 10)
node_c.children.append(leaf5)

new_pruned = []
new_val = alpha_beta(root, 3, float('-inf'), float('inf'), True, new_pruned)
print(f"Updated Root Value: {new_val}")
print(f"Updated Pruned Nodes: {new_pruned}")

Updated Root Value: 3
Updated Pruned Nodes: ['F']


Q4

In [1]:
!pip install ortools
from ortools.sat.python import cp_model

model = cp_model.CpModel()

a = model.NewIntVar(0, 3, 'A')
b = model.NewIntVar(0, 3, 'B')
c = model.NewIntVar(0, 3, 'C')

model.Add(a != b)
model.Add(b != c)
model.Add(a + b <= 4)

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"A = {solver.Value(a)}, B = {solver.Value(b)}, C = {solver.Value(c)}")

A = 0, B = 1, C = 3


Q5

In [2]:
class VarSolutionPrinter(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__variables = variables

    def on_solution_callback(self):
        for v in self.__variables:
            print(f"{v.Name()} = {self.Value(v)}", end=' ')
        print()

model = cp_model.CpModel()
a = model.NewIntVar(0, 3, 'A')
b = model.NewIntVar(0, 3, 'B')
c = model.NewIntVar(0, 3, 'C')

model.Add(a != b)
model.Add(b != c)
model.Add(a + b <= 4)

solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solution_printer = VarSolutionPrinter([a, b, c])
solver.Solve(model, solution_printer)

A = 1 B = 0 C = 1 
A = 0 B = 1 C = 0 
A = 2 B = 1 C = 0 
A = 2 B = 0 C = 1 
A = 1 B = 2 C = 1 
A = 0 B = 2 C = 1 
A = 0 B = 2 C = 0 
A = 1 B = 2 C = 0 
A = 1 B = 2 C = 3 
A = 0 B = 2 C = 3 
A = 0 B = 1 C = 3 
A = 1 B = 0 C = 3 
A = 1 B = 0 C = 2 
A = 0 B = 1 C = 2 
A = 2 B = 1 C = 2 
A = 2 B = 0 C = 2 
A = 2 B = 0 C = 3 
A = 2 B = 1 C = 3 
A = 3 B = 1 C = 3 
A = 3 B = 0 C = 3 
A = 3 B = 0 C = 2 
A = 3 B = 1 C = 2 
A = 3 B = 0 C = 1 
A = 3 B = 1 C = 0 
A = 0 B = 3 C = 0 
A = 0 B = 3 C = 1 
A = 0 B = 3 C = 2 
A = 1 B = 3 C = 2 
A = 1 B = 3 C = 0 
A = 1 B = 3 C = 1 


<CpSolverStatus.OPTIMAL: 4>

Q6

In [3]:
model = cp_model.CpModel()

x = model.NewIntVar(0, 20, 'x')
y = model.NewIntVar(0, 20, 'y')
z = model.NewIntVar(0, 20, 'z')

model.Add(x + 2 * y + z <= 20)
model.Add(3 * x + y <= 18)

model.Maximize(4 * x + 2 * y + z)

solver = cp_model.CpSolver()
status = solver.Solve(model)

print(f"Optimal Value: {solver.ObjectiveValue()}")
print(f"x = {solver.Value(x)}, y = {solver.Value(y)}, z = {solver.Value(z)}")

Optimal Value: 38.0
x = 6, y = 0, z = 14


Q7

In [4]:
model = cp_model.CpModel()
n = 4
queens = [model.NewIntVar(0, n - 1, f'q{i}') for i in range(n)]

model.AddAllDifferent(queens)
model.AddAllDifferent(queens[i] + i for i in range(n))
model.AddAllDifferent(queens[i] - i for i in range(n))

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    for i in range(n):
        row = ["_"] * n
        row[solver.Value(queens[i])] = "Q"
        print(" ".join(row))

_ _ Q _
Q _ _ _
_ _ _ Q
_ Q _ _
